# CSE4002 – Social Network Analysis Practicals 1–8\n**Complete exam reference notebook**\n

## Practical 1 — Reddit interaction network and MongoDB
**Aim from the report:** Build a directed, weighted Reddit interaction network from the SNAP dataset, store nodes and edges in MongoDB, calculate in-degree/out-degree/density, visualize the graph, and identify top 10 users by in-degree.

**Report results:** 14,367 nodes, 151,289 edges, density 0.000733. Use these as report values; dataset versions or preprocessing may change results.

The exact original Practical 1 source code is not present in this notebook, so the following is a reference implementation of the reported workflow, not a verbatim transcription.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

STUDENT_ID = "23CE021"
FULL_NAME = "Nishit P Desai"
print("Student ID:", STUDENT_ID, "|", FULL_NAME)

# Replace with the edge-list file supplied/used in your practical.
# Expected format: source_user target_user [weight]
G = nx.read_weighted_edgelist(
    "reddit_edges.txt",
    create_using=nx.DiGraph(),
    nodetype=str
)

n, m = G.number_of_nodes(), G.number_of_edges()
density = nx.density(G) if n > 1 else 0
print("Nodes:", n, "Edges:", m, "Density:", density)
print("Average in-degree:", sum(dict(G.in_degree()).values())/n if n else 0)
print("Average out-degree:", sum(dict(G.out_degree()).values())/n if n else 0)

top10 = sorted(G.in_degree(), key=lambda item: item[1], reverse=True)[:10]
print("Top 10 users by in-degree:", top10)

plt.figure(figsize=(12,8))
sample = list(G.nodes())[:100]
H = G.subgraph(sample)
pos = nx.spring_layout(H, seed=42)
nx.draw_networkx(H, pos, with_labels=False, node_size=25, arrows=True)
plt.title(f"Reddit Interaction Network - Student ID: {STUDENT_ID}")
plt.axis("off"); plt.tight_layout(); plt.show()

# Optional MongoDB persistence (requires pymongo and a configured URI):
# from pymongo import MongoClient
# client = MongoClient(os.environ["MONGODB_URI"])
# db = client["reddit_network"]
# db.nodes.insert_many([{"user_id": n} for n in G.nodes()])
# db.edges.insert_many([{"source": u, "target": v, **d}
#                       for u, v, d in G.edges(data=True)])


## Common setup

In [ ]:
STUDENT_ID = "23CE021"
FULL_NAME = "Nishit P Desai"

import os
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

print(f"Student ID: {STUDENT_ID}")
print(f"Student Name: {FULL_NAME}")


## Practical 5 — Bipartite network and one-mode projections
**Aim:** Build an affiliation graph (people ↔ projects), project each node set into a one-mode graph, and compare node/edge counts, average degree, and clustering coefficient.

In [ ]:
OUTPUT_DIR = "practical5_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

affiliations = [
    ("Alice", "Project1"), ("Alice", "Project2"),
    ("Bob", "Project1"), ("Bob", "Project3"),
    ("Charlie", "Project1"), ("Charlie", "Project2"),
    ("David", "Project2"), ("David", "Project3"),
    ("Emma", "Project3"), ("Emma", "Project4"),
    ("Frank", "Project4"), ("Frank", "Project2"),
]

B = nx.Graph()
people, projects = set(), set()
for person, project in affiliations:
    B.add_node(person, bipartite=0, node_type="Person")
    B.add_node(project, bipartite=1, node_type="Project")
    B.add_edge(person, project)
    people.add(person)
    projects.add(project)

print("Original bipartite graph")
print("Nodes:", B.number_of_nodes(), "Edges:", B.number_of_edges())
print("People:", len(people), "Projects:", len(projects))

plt.figure(figsize=(10, 6))
pos = nx.bipartite_layout(B, people)
colors = ["skyblue" if n in people else "orange" for n in B.nodes()]
nx.draw_networkx(B, pos=pos, with_labels=True, node_color=colors,
                 node_size=1100, font_size=9, edge_color="gray")
plt.title(f"Bipartite Affiliation Network - {STUDENT_ID}")
plt.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "practical5_bipartite_graph.png"), dpi=300)
plt.show()

people_projection = nx.bipartite.projected_graph(B, people)
project_projection = nx.bipartite.projected_graph(B, projects)

def calculate_metrics(graph):
    n, m = graph.number_of_nodes(), graph.number_of_edges()
    return {
        "Total Nodes": n,
        "Total Edges": m,
        "Average Degree": round(2 * m / n, 4) if n else 0,
        "Clustering Coefficient": round(nx.average_clustering(graph), 4) if n else 0,
    }

for name, graph, color in [
    ("People One-Mode Projection", people_projection, "skyblue"),
    ("Project One-Mode Projection", project_projection, "orange"),
]:
    print(f"\n{name}: {calculate_metrics(graph)}")
    plt.figure(figsize=(8, 6))
    pos = nx.spring_layout(graph, seed=42)
    nx.draw_networkx(graph, pos=pos, with_labels=True, node_color=color,
                     node_size=1400, font_size=10, edge_color="gray")
    plt.title(f"{name} - {STUDENT_ID}")
    plt.axis("off")
    plt.tight_layout()
    filename = "practical5_people_projection.png" if graph is people_projection else "practical5_project_projection.png"
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=300)
    plt.show()

metrics_df = pd.DataFrame({
    "Original Bipartite Graph": calculate_metrics(B),
    "Projected People Graph": calculate_metrics(people_projection),
    "Projected Project Graph": calculate_metrics(project_projection),
})
display(metrics_df)
metrics_df.to_csv(os.path.join(OUTPUT_DIR, "practical5_metrics.csv"))
nx.write_edgelist(people_projection, os.path.join(OUTPUT_DIR, "practical5_people_projection_edges.txt"), data=False)
nx.write_edgelist(project_projection, os.path.join(OUTPUT_DIR, "practical5_project_projection_edges.txt"), data=False)
print("Practical 5 completed.")


### Practical 5 viva points
- Bipartite graph: nodes are divided into two disjoint sets; edges run only between sets.
- One-mode projection connects same-set nodes that share a neighbor.
- People projection means people collaborated on a project; project projection means projects share participants.
- Average degree in an undirected graph is `2E/N`. Bipartite graphs have no triangles, so their clustering coefficient is 0.


## Practical 6 — Random and scale-free networks; centrality comparison
**Aim:** Generate an Erdős–Rényi random graph and a Barabási–Albert scale-free graph; calculate five centrality measures and compare top-ranked nodes.

In [ ]:
OUTPUT_DIR = "practical6_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
N, SEED = 100, 42
random_graph = nx.gnp_random_graph(N, 0.05, seed=SEED)
scale_free_graph = nx.barabasi_albert_graph(N, 3, seed=SEED)

for graph_name, graph, filename in [
    ("Random Network", random_graph, "random_network.png"),
    ("Scale-Free Network", scale_free_graph, "scale_free_network.png"),
]:
    print(f"\n{graph_name}: Nodes={graph.number_of_nodes()}, Edges={graph.number_of_edges()}")
    plt.figure(figsize=(9, 6))
    pos = nx.spring_layout(graph, seed=SEED)
    nx.draw_networkx(graph, pos=pos, with_labels=False, node_size=45,
                     edge_color="gray", node_color="skyblue")
    plt.title(f"{graph_name} - {STUDENT_ID}")
    plt.axis("off")
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=300)
    plt.show()
    plt.close()

def calculate_centralities(graph):
    return {
        "Degree Centrality": nx.degree_centrality(graph),
        "Betweenness Centrality": nx.betweenness_centrality(graph),
        "Closeness Centrality": nx.closeness_centrality(graph),
        "Eigenvector Centrality": nx.eigenvector_centrality(graph, max_iter=1000, tol=1e-6),
        "PageRank": nx.pagerank(graph, alpha=0.85),
    }

random_centralities = calculate_centralities(random_graph)
scale_centralities = calculate_centralities(scale_free_graph)

comparison_rows = []
for measure in random_centralities:
    top_random = sorted(random_centralities[measure].items(), key=lambda x: x[1], reverse=True)[:5]
    top_scale = sorted(scale_centralities[measure].items(), key=lambda x: x[1], reverse=True)[:5]
    print(f"\n{measure}")
    print("Random top 5:", [(n, round(v, 4)) for n, v in top_random])
    print("Scale-free top 5:", [(n, round(v, 4)) for n, v in top_scale])
    comparison_rows.append({
        "Centrality Algorithm": measure,
        "Top Nodes (Random)": str([(n, round(v, 4)) for n, v in top_random]),
        "Top Nodes (Scale-Free)": str([(n, round(v, 4)) for n, v in top_scale]),
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)
comparison_df.to_csv(os.path.join(OUTPUT_DIR, "practical6_top5_comparison.csv"), index=False)

all_rows = []
for network_name, centralities in [("Random Network", random_centralities), ("Scale-Free Network", scale_centralities)]:
    for measure, scores in centralities.items():
        for node, score in scores.items():
            all_rows.append({"Network": network_name, "Centrality Algorithm": measure, "Node": node, "Score": score})
pd.DataFrame(all_rows).to_csv(os.path.join(OUTPUT_DIR, "practical6_all_centrality_scores.csv"), index=False)

for measure in random_centralities:
    safe_name = measure.lower().replace(" ", "_")
    r_scores = list(random_centralities[measure].values())
    s_scores = list(scale_centralities[measure].values())

    plt.figure(figsize=(9, 5))
    plt.hist(r_scores, bins=20, alpha=0.65, label="Random Network")
    plt.hist(s_scores, bins=20, alpha=0.65, label="Scale-Free Network")
    plt.xlabel("Centrality Score"); plt.ylabel("Number of Nodes")
    plt.title(f"{measure} Distribution - {STUDENT_ID}")
    plt.legend(); plt.grid(axis="y", alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"practical6_{safe_name}_distribution.png"), dpi=300)
    plt.show(); plt.close()

    plt.figure(figsize=(9, 5))
    plt.scatter(range(len(r_scores)), sorted(r_scores, reverse=True), label="Random Network", alpha=0.7)
    plt.scatter(range(len(s_scores)), sorted(s_scores, reverse=True), label="Scale-Free Network", alpha=0.7)
    plt.xlabel("Node Rank"); plt.ylabel("Centrality Score")
    plt.title(f"{measure} Ranked Scores - {STUDENT_ID}")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"practical6_{safe_name}_ranked_scores.png"), dpi=300)
    plt.show(); plt.close()

print("Practical 6 completed.")


### Practical 6 viva points
- Random graph `G(n,p)`: each possible edge is independently included with probability `p`.
- Barabási–Albert model: preferential attachment; new nodes tend to connect to already well-connected nodes, producing hubs.
- Degree: direct neighbors. Betweenness: shortest-path bridge role. Closeness: distance to other nodes. Eigenvector: connected to influential nodes. PageRank: importance from incoming links/neighbor importance.
- Typical observation: scale-free graphs have a few hubs and a skewed degree distribution; random graphs are more homogeneous.


## Practical 7 — Motif detection and random-graph comparison
**Aim:** Count triangles, 3-star subgraphs, and 4-cliques in the SNAP Facebook graph and compare against a random graph with the same number of nodes and edges.

In [ ]:
OUTPUT_DIR = "practical7_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
DATA_URL = "https://snap.stanford.edu/data/facebook_combined.txt.gz"
DATA_FILE = os.path.join(OUTPUT_DIR, "facebook_combined.txt")
GZ_FILE = os.path.join(OUTPUT_DIR, "facebook_combined.txt.gz")

if not os.path.exists(DATA_FILE):
    import gzip
    import urllib.request
    print("Downloading SNAP Facebook dataset...")
    urllib.request.urlretrieve(DATA_URL, GZ_FILE)
    with gzip.open(GZ_FILE, "rt", encoding="utf-8") as src, open(DATA_FILE, "w", encoding="utf-8") as dst:
        dst.write(src.read())

G = nx.read_edgelist(DATA_FILE, nodetype=int)
G = nx.Graph(G)
G.remove_edges_from(nx.selfloop_edges(G))
print("Facebook network: nodes =", G.number_of_nodes(), "edges =", G.number_of_edges(),
      "density =", round(nx.density(G), 6))

from itertools import combinations

def count_triangles(graph):
    return sum(nx.triangles(graph).values()) // 3

def count_three_stars(graph):
    # Choose any 3 neighbors of each possible center.
    return sum(degree * (degree - 1) * (degree - 2) // 6 for _, degree in graph.degree())

def count_k4_cliques(graph):
    # Count each 4-clique once using ordered node IDs.
    adjacency = {node: set(graph.neighbors(node)) for node in graph.nodes()}
    total = 0
    for u, v in graph.edges():
        if u > v:
            u, v = v, u
        common = sorted(w for w in adjacency[u] & adjacency[v] if w > v)
        for w, x in combinations(common, 2):
            if graph.has_edge(w, x):
                total += 1
    return total

real_counts = {
    "Triangles (K3)": count_triangles(G),
    "3-Star Structures": count_three_stars(G),
    "4-Node Cliques (K4)": count_k4_cliques(G),
}
print("Facebook motif counts:", real_counts)

R = nx.gnm_random_graph(G.number_of_nodes(), G.number_of_edges(), seed=42)
random_counts = {
    "Triangles (K3)": count_triangles(R),
    "3-Star Structures": count_three_stars(R),
    "4-Node Cliques (K4)": count_k4_cliques(R),
}
print("Random graph motif counts:", random_counts)

rows = []
for motif, real in real_counts.items():
    rand = random_counts[motif]
    rows.append({"Motif Type": motif, "Facebook Count": real, "Random Count": rand,
                 "Observed/Random Ratio": round(real / rand, 4) if rand else None})
results_df = pd.DataFrame(rows)
display(results_df)
results_df.to_csv(os.path.join(OUTPUT_DIR, "practical7_motif_frequency.csv"), index=False)

labels = list(real_counts)
x = list(range(len(labels)))
width = 0.36
plt.figure(figsize=(10, 6))
plt.bar([i - width/2 for i in x], [real_counts[k] for k in labels], width, label="Facebook Graph")
plt.bar([i + width/2 for i in x], [random_counts[k] for k in labels], width, label="Random Graph")
plt.xticks(x, labels, rotation=10); plt.ylabel("Motif Count")
plt.title(f"Motif Frequency: Facebook vs Random Graph - {STUDENT_ID}")
plt.legend(); plt.grid(axis="y", alpha=0.25); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "practical7_motif_comparison.png"), dpi=300)
plt.show(); plt.close()

plt.figure(figsize=(10, 7))
pos = nx.spring_layout(G, seed=42, k=0.15, iterations=30)
nx.draw_networkx(G, pos=pos, with_labels=False, node_size=5, edge_color="gray",
                 width=0.25, node_color="skyblue")
plt.title(f"Facebook Network - {STUDENT_ID}")
plt.axis("off"); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "practical7_facebook_network.png"), dpi=300)
plt.show(); plt.close()
print("Practical 7 completed.")


### Practical 7 viva points
- Motif: a small recurring subgraph pattern.
- Triangle/K3: three nodes mutually connected; indicates triadic closure/tightly knit groups.
- 3-star: one center connected to three leaves; leaves need not connect to each other.
- K4: four nodes with all six possible edges; a highly cohesive group.
- Observed/random ratio greater than 1 means the motif occurs more often in the real network than in the random baseline.


## Practical 8 — Gephi network visualization workflow
**Aim:** Import a large edge list, apply layouts, detect communities, and filter nodes for clearer interpretation.

1. Download the SNAP Facebook edge list (`facebook_combined.txt`).
2. Convert it to CSV with headers `Source,Target` if needed; import in Gephi as an **Edges table**.
3. Set graph type to **Undirected** and verify import statistics.
4. Run **ForceAtlas 2**; optionally compare with **Yifan Hu**. Layout changes node positions, not graph relationships.
5. Run **Modularity** (report used resolution 1.0); color nodes by **Modularity Class** in Appearance.
6. Apply **Degree Range** filter, minimum degree 5; inspect the remaining network.
7. Save the Gephi project and capture screenshots of layout, communities, and filtering.

**Report figures to know:** the supplied report describes 4,039 nodes and 88,234 edges; Modularity found 14 communities with score 0.835; degree ≥5 filtering left 3,674 nodes and 87,360 edges. These are report-specific results and can vary with dataset/import/settings.


## Quick practical-exam recall
| Practical | Core pipeline | Key result/interpretation |
|---|---|---|
| 5 | Build bipartite → project both sets → calculate metrics | Projection connects same-type nodes sharing an affiliation |
| 6 | Generate ER + BA graphs → five centralities → compare top 5 and distributions | BA has hubs; metrics identify different notions of importance |
| 7 | Load Facebook graph → count motifs → compare with random baseline | Motif enrichment reveals cohesive/local structure |
| 8 | Import in Gephi → layout → modularity → degree filter | Layout is visual; modularity colors communities; filter reduces clutter |


## Practical 2 — Twitter/X interaction network
**Aim:** Retrieve tweet data and model user interactions using mentions. The report uses X API credentials from environment variables; live API access may be unavailable, so use a prepared public dataset when needed.


In [ ]:
# Student identity must appear in source, console, and plot titles.
STUDENT_ID = "23CE021"
FULL_NAME = "Nishit P Desai"
print("Student ID:", STUDENT_ID)
print("Student Name:", FULL_NAME)

import os, json
import networkx as nx
import matplotlib.pyplot as plt

tweets_data = []  # Populate from API response or your permitted dataset.
G = nx.DiGraph()

for tweet in tweets_data:
    author = tweet.get("author_id")
    entities = tweet.get("entities", {}) or {}
    mentions = entities.get("mentions", []) or []
    if author:
        G.add_node(str(author))
    for mention in mentions:
        target = mention.get("id")
        if author and target and str(author) != str(target):
            source, target = str(author), str(target)
            if G.has_edge(source, target):
                G[source][target]["weight"] += 1
            else:
                G.add_edge(source, target, weight=1)

with open("tweets_data.json", "w", encoding="utf-8") as f:
    json.dump(tweets_data, f, indent=4, ensure_ascii=False)

n, m = G.number_of_nodes(), G.number_of_edges()
density = nx.density(G) if n > 1 else 0
avg_in = sum(dict(G.in_degree()).values()) / n if n else 0
avg_out = sum(dict(G.out_degree()).values()) / n if n else 0
print("Nodes:", n, "Edges:", m, "Density:", density)
print("Average in-degree:", avg_in, "Average out-degree:", avg_out)

if n:
    H = G.subgraph(list(G.nodes())[:100]).copy()
    pos = nx.spring_layout(H, seed=42)
    nx.draw_networkx(H, pos, with_labels=False, node_size=40, arrows=True)
    plt.title(f"Twitter Mention Network - Student ID: {STUDENT_ID}")
    plt.axis("off"); plt.tight_layout()
    plt.savefig("practical2_network.png", dpi=300); plt.show()

with open("network_edges.json", "w", encoding="utf-8") as f:
    json.dump([{"source": u, "target": v, **d} for u,v,d in G.edges(data=True)], f, indent=4)


### Practical 2 viva
- Nodes represent users; directed edges represent source-to-target mentions.
- Edge weight counts repeated mentions.
- In-degree counts incoming edges; out-degree counts outgoing edges.
- API limitations include rate limits, pagination, authentication, latency, and access restrictions.


## Practical 3 — Centrality comparison and scalability
**Aim:** Compare PageRank, Katz, Eigenvector, Closeness, and Betweenness on random and scale-free networks, recording execution time.


In [ ]:
import time, os
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

STUDENT_ID = "23CE021"
FULL_NAME = "Nishit P Desai"
print("Student ID:", STUDENT_ID, "|", FULL_NAME)
OUTPUT_DIR = "practical3_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

N = 500
networks = {
    "Random Network": nx.gnp_random_graph(N, 0.02, seed=42),
    "Scale-Free Network": nx.barabasi_albert_graph(N, 3, seed=42)
}

def get_centrality_algorithms(graph):
    return {
        "PageRank": lambda: nx.pagerank(graph, alpha=0.85),
        "Katz": lambda: nx.katz_centrality(graph, alpha=0.01, beta=1.0, max_iter=1000, tol=1e-6),
        "Eigenvector": lambda: nx.eigenvector_centrality(graph, max_iter=1000, tol=1e-6),
        "Closeness": lambda: nx.closeness_centrality(graph),
        "Betweenness": lambda: nx.betweenness_centrality(graph, normalized=True)
    }

rows = []
for network_name, graph in networks.items():
    for metric, algorithm in get_centrality_algorithms(graph).items():
        start = time.perf_counter()
        try:
            scores = algorithm()
            elapsed = time.perf_counter() - start
            avg = sum(scores.values()) / len(scores) if scores else 0
            status = "Success"
        except Exception as e:
            elapsed, avg, status = time.perf_counter()-start, None, f"Failed: {e}"
        rows.append({"Network": network_name, "Centrality": metric,
                     "Average Score": avg, "Execution Time (sec)": elapsed, "Status": status})

results_df = pd.DataFrame(rows)
display(results_df)
results_df.to_csv(os.path.join(OUTPUT_DIR, "practical3_centrality_results.csv"), index=False)

sizes = [100, 200, 300, 500]
scale_rows = []
for n in sizes:
    gs = {"Random Network": nx.gnp_random_graph(n, 0.03, seed=42),
          "Scale-Free Network": nx.barabasi_albert_graph(n, 3, seed=42)}
    for name, graph in gs.items():
        for metric, algorithm in get_centrality_algorithms(graph).items():
            start = time.perf_counter()
            try:
                algorithm(); status = "Success"
            except Exception as e:
                status = f"Failed: {e}"
            scale_rows.append({"Nodes":n, "Network":name, "Centrality":metric,
                              "Execution Time (sec)":time.perf_counter()-start, "Status":status})
scalability_df = pd.DataFrame(scale_rows)
scalability_df.to_csv(os.path.join(OUTPUT_DIR, "practical3_scalability_results.csv"), index=False)
for metric in scalability_df["Centrality"].unique():
    subset = scalability_df[scalability_df["Centrality"] == metric]
    for name in subset["Network"].unique():
        data = subset[subset["Network"] == name].sort_values("Nodes")
        plt.plot(data["Nodes"], data["Execution Time (sec)"], marker="o", label=name)
    plt.title(f"{metric} Scalability - Student ID: {STUDENT_ID}")
    plt.xlabel("Number of Nodes"); plt.ylabel("Execution Time (seconds)")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
    plt.close()


### Practical 3 viva
- Random graph: each possible edge is independently included with probability p.
- Scale-free graph: preferential attachment creates hubs.
- PageRank/Katz/Eigenvector measure importance; closeness measures distance reach; betweenness measures bridge/intermediary role.
- The report tested NetworkX only; it does not establish a direct performance comparison with SNAP or igraph.


## Practical 4 — SIS and SIR epidemic diffusion
**Aim:** Simulate epidemic spread on the Facebook network and compare spread from high- and low-degree starting nodes.


In [ ]:
import os, gzip, random, urllib.request
import networkx as nx
import matplotlib.pyplot as plt

STUDENT_ID = "23CE021"
FULL_NAME = "Nishit P Desai"
print("Student ID:", STUDENT_ID, "|", FULL_NAME)

DATASET_URL = "https://snap.stanford.edu/data/facebook_combined.txt.gz"
COMPRESSED_FILE, DATASET_FILE = "facebook_combined.txt.gz", "facebook_combined.txt"
if not os.path.exists(DATASET_FILE):
    urllib.request.urlretrieve(DATASET_URL, COMPRESSED_FILE)
    with gzip.open(COMPRESSED_FILE, "rb") as src, open(DATASET_FILE, "wb") as dst:
        dst.write(src.read())
G = nx.read_edgelist(DATASET_FILE, nodetype=int, create_using=nx.Graph())
print("Nodes:", G.number_of_nodes(), "Edges:", G.number_of_edges(), "Connected:", nx.is_connected(G))

BETA, GAMMA, TIME_STEPS, INITIAL_SEED = 0.05, 0.02, 30, 42

def simulate_sir(graph, beta=BETA, gamma=GAMMA, steps=TIME_STEPS, seed=INITIAL_SEED, initial_node=None):
    rng = random.Random(seed); nodes = list(graph.nodes())
    if not nodes: raise ValueError("The network has no nodes.")
    if initial_node is None: initial_node = rng.choice(nodes)
    infected, recovered = {initial_node}, set()
    history = {"Susceptible": [], "Infected": [], "Recovered": []}
    for _ in range(steps):
        susceptible = set(nodes) - infected - recovered
        for key, value in [("Susceptible",len(susceptible)),("Infected",len(infected)),("Recovered",len(recovered))]:
            history[key].append(value)
        new_infected, new_recovered = set(), set()
        for node in infected:
            for neighbor in graph.neighbors(node):
                if neighbor in susceptible and rng.random() < beta: new_infected.add(neighbor)
            if rng.random() < gamma: new_recovered.add(node)
        infected = (infected - new_recovered) | new_infected
        recovered |= new_recovered
    return history

def simulate_sis(graph, beta=BETA, gamma=GAMMA, steps=TIME_STEPS, seed=INITIAL_SEED, initial_node=None):
    rng = random.Random(seed); nodes = list(graph.nodes())
    if not nodes: raise ValueError("The network has no nodes.")
    if initial_node is None: initial_node = rng.choice(nodes)
    infected = {initial_node}; history = {"Susceptible": [], "Infected": []}
    for _ in range(steps):
        susceptible = set(nodes) - infected
        history["Susceptible"].append(len(susceptible)); history["Infected"].append(len(infected))
        new_infected, new_recovered = set(), set()
        for node in infected:
            for neighbor in graph.neighbors(node):
                if neighbor in susceptible and rng.random() < beta: new_infected.add(neighbor)
            if rng.random() < gamma: new_recovered.add(node)
        # SIS: recovered nodes return to susceptible state
        infected = (infected - new_recovered) | new_infected
    return history

sir_results, sis_results = simulate_sir(G), simulate_sis(G)
for results, model in [(sir_results,"SIR"),(sis_results,"SIS")]:
    for label, values in results.items(): plt.plot(values, label=label)
    plt.title(f"{model} Epidemic Model - Student ID: {STUDENT_ID}")
    plt.xlabel("Time Steps"); plt.ylabel("Number of Users"); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f"practical4_{model.lower()}_model.png", dpi=300); plt.show(); plt.close()

degrees = dict(G.degree())
high_node, low_node = max(degrees, key=degrees.get), min(degrees, key=degrees.get)
high_spread = simulate_sir(G, initial_node=high_node)["Infected"]
low_spread = simulate_sir(G, initial_node=low_node)["Infected"]
plt.plot(high_spread, label="High-degree starting node")
plt.plot(low_spread, label="Low-degree starting node")
plt.title(f"Hub Influence Analysis - Student ID: {STUDENT_ID}")
plt.xlabel("Time Steps"); plt.ylabel("Currently Infected Users"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("practical4_hub_influence.png", dpi=300); plt.show(); plt.close()
print("High-degree node:", high_node, "degree:", degrees[high_node])
print("Low-degree node:", low_node, "degree:", degrees[low_node])


### Practical 4 viva
- SIS: Susceptible → Infected → Susceptible; recovered individuals can become infected again.
- SIR: Susceptible → Infected → Recovered; recovered individuals are removed from further infection in the model.
- Beta is infection probability per susceptible neighbor contact; gamma is recovery probability per time step.
- Hubs can accelerate spread because they have many neighbors.
